# **任务21 Transformer 解码器 | Transformer - Decoder**

这个任务主要为了体现解码器的功能。

先前我们解释了编码器的结构，对于decoder则要复杂一些，它同样引入了注意力机制、位置编码和归一化和残差连接，但在某些特征上它更像循环神经网络，只不过不再是一个单一细胞作为传递特征的介质，而是使用一整个单层残差感知机网络。

Transformer decoder 本质上是一个续写结构，区别于RNN的一次性定长输出，解码器可以循环输出，所以需要标识符来截断。这一点很想我们之前 **任务9** 的滑动窗口，不同的是它的输入不是一个限定窗口中的内容，而是将用户输入内容和自身先前输出内容作为下一次计算的输入，本质上都是在续写。

## 1. **Transformer编码器层**

实例化方法 `nn.TransformerDecoderLayer(d_model=emb_dim, nhead=nhead, batch_first=True)`

使用 `nn.TransformerDecoder(decoder_layer, num_layers=num_layers)` 与先前的编码器封装结构作用相同。

解码器层有两个输入，一个是 `memory`，代表的是用户的输入；一个是 `tgt` 是模型的逐步输出的一部分，在模型第一次计算的时候，它使用 `开始符` 的嵌入向量作为输入。

In [18]:
import torch
import torch.nn as nn

batch_size = 8
seq_num = 4
emb_dim = 16
nhead = 8
num_layers = 2

decoder_layer = nn.TransformerDecoderLayer(d_model=emb_dim, nhead=nhead, batch_first=True)
decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)

memory = torch.ones(batch_size, 10, emb_dim)
tgt = torch.ones(batch_size, 1, emb_dim)

print('输入张量：', memory.shape)
# 续写5个内容
for i in range(5):
    output_tensor = decoder(tgt, memory)
    # 只取最后一个时间步的输出 [8, n, 16] → [8, 16]
    last_output = output_tensor[:, -1, :]
    # 将序列维度补充回去，不然cat方法拼接不了
    last_output = last_output.unsqueeze(dim=1)
    print(f'第{i}次的输出：', last_output.shape)
    # 拼接回原有的数据作为新的 tgt
    tgt = torch.cat([tgt, last_output], dim=1)

    print(f'第{i}次的tgt张量：', tgt.shape)

输入张量： torch.Size([8, 10, 16])
第0次的输出： torch.Size([8, 1, 16])
第0次的tgt张量： torch.Size([8, 2, 16])
第1次的输出： torch.Size([8, 1, 16])
第1次的tgt张量： torch.Size([8, 3, 16])
第2次的输出： torch.Size([8, 1, 16])
第2次的tgt张量： torch.Size([8, 4, 16])
第3次的输出： torch.Size([8, 1, 16])
第3次的tgt张量： torch.Size([8, 5, 16])
第4次的输出： torch.Size([8, 1, 16])
第4次的tgt张量： torch.Size([8, 6, 16])


Transformer解码器的输入输出比较复杂，这里我直接将带特征的序列数据作为输入和续写目标，但在实际任务中，输入可能需要经过嵌入层，并且memory和tgt的嵌入层对于不同任务可能需要两个嵌入层。比如英文翻译中文的机器翻译任务，英文作为memory输入拥有一个嵌入层，中文作为tgt的输入拥有一个嵌入层。